# Semana 01 · Tracking MLOps con MLflow en Databricks

En esta práctica vas a entrenar modelos **muy sencillos** para centrarte en el ciclo MLOps: datos, parámetros, métricas, artefactos, comparación y una decisión reproducible. El objetivo no es ganar una competición de machine learning, sino dejar evidencia clara en MLflow.


## Mapa y criterios de terminación

Al acabar debes poder enseñar:

1. un experimento de MLflow con varios runs comparables;
2. tags que identifiquen tu alias, lote y propósito docente;
3. parámetros y métricas de validación/test;
4. un modelo ganador guardado como artefacto;
5. un pequeño smoke test de inferencia.


## 0. Dependencias

En Databricks, ejecuta `%pip` al principio. Si el entorno reinicia Python, continúa desde la celda de imports.


In [0]:
%pip install -q --upgrade "mlflow[databricks]" "scikit-learn" "pandas<3" "matplotlib"

In [0]:
dbutils.library.restartPython()  # reinicia Python para que cargue las librerías recién instaladas

## 1. Configuración e identidad del lote

Usa rutas de Databricks (`/dbfs/...`) para artefactos pequeños que quieras inspeccionar fuera de MLflow.


In [0]:

import tempfile  # para crear una carpeta temporal si /dbfs no está disponible
import time  # para medir cuánto tarda el entrenamiento
import uuid  # para generar un identificador aleatorio de lote
from pathlib import Path  # para trabajar con rutas de ficheros
import json  # para guardar diccionarios como ficheros .json 

import mlflow  # librería de tracking: experimentos, runs, métricas, modelos
import numpy as np  # cálculo numérico
import pandas as pd  # tablas de datos (DataFrames)
from databricks.sdk import WorkspaceClient  # cliente oficial para manejar carpetas del workspace
from sklearn.datasets import load_wine  # dataset pequeño de vinos
from sklearn.dummy import DummyClassifier  # modelo "tonto" que sirve de baseline
from sklearn.ensemble import RandomForestClassifier  # bosque aleatorio
from sklearn.linear_model import LogisticRegression  # regresión logística
from sklearn.metrics import accuracy_score, classification_report, f1_score  # métricas
from sklearn.model_selection import train_test_split  # para partir los datos
from sklearn.pipeline import Pipeline  # para encadenar pasos (escalar + modelo)
from sklearn.preprocessing import StandardScaler  # normaliza cada columna

SEED = 42  # semilla fija: mismos resultados cada vez
STUDENT_ALIAS = "gonzalo-s01"  # TODO 1: tu alias estable para filtrar tus runs
BATCH_ID = f"s01-{uuid.uuid4().hex[:8]}"  # id único de esta ejecución
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-tracking"  # experimento en el workspace

try:  # intentamos usar DBFS como pide la práctica
    ARTIFACT_DIR = Path("/dbfs/tmp/muiaap_s01_tracking")  # ruta en DBFS
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)  # crea la carpeta
    test_file = ARTIFACT_DIR / "_write_test.txt"  # fichero de prueba
    test_file.write_text("ok", encoding="utf-8")  # probamos a ESCRIBIR (en serverless falla aquí)
    test_file.unlink()  # borramos el fichero de prueba
except Exception:  # si DBFS está bloqueado...
    ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="muiaap_s01_"))  # ...usamos una carpeta temporal

WorkspaceClient().workspace.mkdirs("/Shared/muiaap-operacion-modelos")  # NUEVO: crea la carpeta del experimento si no existe
mlflow.set_experiment(EXPERIMENT_NAME)  # activa (o crea) el experimento
print({"batch_id": BATCH_ID, "alias": STUDENT_ALIAS, "artifact_dir": str(ARTIFACT_DIR)})  # mostramos la configuración

## 2. Datos pequeños y contrato mínimo

Usaremos `load_wine` de scikit-learn: no descarga datos, es rápido y suficiente para practicar tracking.


In [0]:
wine = load_wine(as_frame=True)
data = wine.frame.rename(columns={"target": "quality_class"})
FEATURES = list(wine.feature_names)
TARGET = "quality_class"

assert TARGET in data.columns
assert data[FEATURES].isna().sum().sum() == 0
assert data[TARGET].nunique() == 3

display(data.head())
print(data.shape, data[TARGET].value_counts().sort_index().to_dict())

## 3. Separar train/validación/test

La celda ya trae una partición sencilla para que la práctica sea ejecutable en Databricks.

**TODO 2:** revisa si el reparto 70/15/15 mantiene todas las clases en validación y test. Decide si cambiarías el porcentaje de test en un proyecto real, porque la evaluación debe ser comparable y suficientemente estable. Inspecciona `train_test_split` y el parámetro `stratify`. Verifica con los `assert` y los conteos por clase.


In [0]:
X = data[FEATURES]
y = data[TARGET]

# Partición base 70/15/15. TODO 2: revisa los tamaños y conteos de clase antes de aceptar este split.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

assert len(X_train) + len(X_valid) + len(X_test) == len(data)
assert y_valid.nunique() == 3, "Falta alguna clase en validación"  # ✅ TODO 2: las 3 clases están en validación
assert y_test.nunique() == 3, "Falta alguna clase en test"  # ✅ TODO 2: las 3 clases están en test
print({"train": len(X_train), "valid": len(X_valid), "test": len(X_test)})
print("valid classes:", y_valid.value_counts().sort_index().to_dict())
print("test classes:", y_test.value_counts().sort_index().to_dict())


## 4. Candidatos simples

Mantén pocos candidatos para que el foco esté en MLflow. Un baseline, una regresión logística y un bosque pequeño son suficientes.

**TODO 3:** revisa los parámetros y decide si cambiarías uno. Importa porque los parámetros deben explicar diferencias entre runs. Inspecciona la API del estimador elegido y verifica que cada candidato tiene nombre y modelo.


In [0]:
CANDIDATES = [
    {
        "name": "baseline_most_frequent",
        "model": DummyClassifier(strategy="most_frequent"),
        "params": {"family": "baseline", "strategy": "most_frequent"},
    },
    {
        "name": "logistic_regression",
        "model": Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=500, random_state=SEED)),
        ]),
        "params": {"family": "linear", "max_iter": 500},
    },
    {
        "name": "random_forest_small",
        "model": RandomForestClassifier(n_estimators=60, max_depth=4, random_state=SEED),
        "params": {"family": "trees", "n_estimators": 60, "max_depth": 4},
    },
]

assert all("name" in c and "model" in c for c in CANDIDATES)
[c["name"] for c in CANDIDATES]

## 5. Registrar runs comparables

Cada run debe contar la misma historia: identidad, parámetros, métrica de validación y artefactos mínimos.

**TODO 4:** completa o revisa los tags. Importa porque después filtrarás sólo tus runs. Inspecciona `mlflow.set_tags`, `mlflow.log_params`, `mlflow.log_metrics` y `mlflow.sklearn.log_model`. Verifica que `run_infos` contiene un `model_uri` por candidato.


In [0]:
run_infos = []  # aquí guardamos un resumen de cada run

for candidate in CANDIDATES:  # repetimos lo mismo para cada modelo
    with mlflow.start_run(run_name=f"{BATCH_ID}-{candidate['name']}") as run:  # abre un run nuevo
        mlflow.set_tags({  # TODO 4: etiquetas de identidad del run
            "course": "operacion-modelos",  # asignatura
            "week": "01",  # semana
            "student.alias": STUDENT_ALIAS,  # quién lo ha hecho
            "batch_id": BATCH_ID,  # de qué ejecución viene
            "purpose": "guided-mlflow-tracking",  # para qué es
            "candidate": candidate["name"],  # qué modelo es
        })
        mlflow.log_params(candidate["params"])  # guarda los hiperparámetros
        mlflow.log_param("feature_count", len(FEATURES))  # nº de variables
        mlflow.log_param("train_rows", len(X_train))  # nº de filas de train
        mlflow.log_param("validation_rows", len(X_valid))  # nº de filas de validación

        start = time.perf_counter()  # empieza el cronómetro
        model = candidate["model"]  # cogemos el modelo
        model.fit(X_train, y_train)  # lo entrenamos con train
        valid_pred = model.predict(X_valid)  # predecimos validación
        latency_ms = (time.perf_counter() - start) * 1000  # tiempo en milisegundos

        metrics = {  # métricas a comparar entre runs
            "validation.accuracy": accuracy_score(y_valid, valid_pred),  # % de aciertos
            "validation.f1_macro": f1_score(y_valid, valid_pred, average="macro"),  # F1 medio entre clases
            "train_latency_ms": latency_ms,  # tiempo de entrenar + predecir
        }
        mlflow.log_metrics(metrics)  # guarda las métricas

        report = classification_report(y_valid, valid_pred, output_dict=True, zero_division=0)  # informe por clase
        report_path = ARTIFACT_DIR / f"{candidate['name']}_validation_report.json"  # ruta del fichero
        report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")  # escribe el informe
        mlflow.log_artifact(str(report_path), artifact_path="validation")  # lo sube al run
        model_info = mlflow.sklearn.log_model(  # guarda el modelo en el run
            model,  # el modelo entrenado
            name="model",  # nombre de la carpeta del modelo
            input_example=X_train.head(3),  # filas de ejemplo
            skops_trusted_types=["sklearn.tree._tree.Tree"],  # ESTA ES LA LÍNEA QUE ARREGLA EL ERROR
        )

        run_infos.append({  # resumen de este run
            "run_id": run.info.run_id,  # id del run
            "name": candidate["name"],  # nombre del modelo
            "model_uri": model_info.model_uri,  # dirección del modelo para cargarlo luego
            **metrics,  # y sus métricas
        })

runs_df = pd.DataFrame(run_infos).sort_values("validation.f1_macro", ascending=False)  # tabla ordenada por F1
display(runs_df)  # la mostramos
assert len(runs_df) == len(CANDIDATES)  # un run por candidato
assert runs_df["model_uri"].notna().all()  # todos tienen modelo guardado

## 6. Buscar tus runs y elegir ganador sin mirar test

**TODO 5:** filtra por `student.alias` y `batch_id`. Importa porque en un workspace compartido hay runs de otras personas. Inspecciona `mlflow.search_runs` y verifica que no aparecen runs de otro lote.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
filter_string = f"tags.student.alias = '{STUDENT_ALIAS}' and tags.batch_id = '{BATCH_ID}'"
searched = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=filter_string,
    order_by=["metrics.validation.f1_macro DESC"],
)

display(searched[["run_id", "tags.mlflow.runName", "metrics.validation.f1_macro", "metrics.validation.accuracy"]])
assert set(searched["tags.batch_id"]) == {BATCH_ID}

best_row = searched.iloc[0]
BEST_RUN_ID = best_row["run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"
print({"best_run_id": BEST_RUN_ID, "best_model_uri": BEST_MODEL_URI})

## 7. Evaluar test una sola vez

Test se usa después de elegir. Esto evita optimizar contra la evidencia final.

**TODO 6:** registra métricas de test en el run ganador. Importa porque la decisión queda trazada en el mismo run. Inspecciona `mlflow.start_run(run_id=...)` y verifica que las métricas `test.*` aparecen en MLflow.


In [0]:
best_model = mlflow.sklearn.load_model(BEST_MODEL_URI)
test_pred = best_model.predict(X_test)
test_metrics = {
    "test.accuracy": accuracy_score(y_test, test_pred),
    "test.f1_macro": f1_score(y_test, test_pred, average="macro"),
}

with mlflow.start_run(run_id=BEST_RUN_ID):
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({"selection.stage": "tested", "selection.reason": "best_validation_f1_macro"})

print(test_metrics)

## 8. Smoke test de inferencia

Comprueba que un consumidor puede cargar el modelo por URI y predecir pocas filas. No levantamos API local ni usamos CLI: eso llegará en semanas posteriores.


In [0]:
sample = X_test.head(3)
predictions = best_model.predict(sample)
result = pd.DataFrame({"prediction": predictions}, index=sample.index)
display(pd.concat([sample.reset_index(drop=True), result.reset_index(drop=True)], axis=1))
assert len(predictions) == 3
assert set(predictions).issubset(set(y.unique()))

## Entregable

Incluye en tu ficha: `EXPERIMENT_NAME`, `BATCH_ID`, `BEST_RUN_ID`, la métrica de validación usada para elegir, la métrica de test y una captura del smoke test.


In [0]:
print({  # resumen para la ficha de entrega
    "EXPERIMENT_NAME": EXPERIMENT_NAME,  # experimento
    "BATCH_ID": BATCH_ID,  # lote
    "BEST_RUN_ID": BEST_RUN_ID,  # run ganador
    "validation.f1_macro": round(float(best_row["metrics.validation.f1_macro"]), 4),  # métrica usada para elegir
    "test.f1_macro": round(test_metrics["test.f1_macro"], 4),  # métrica final de test
})